In [183]:
from vespa.package import Field
%load_ext autoreload
%autoreload 2
import json
import mycode.vap as vap

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# The goal is to have a demo application that has 1M docs
# - numeric field
# - single dimension embedding field for the nearestNeighbor search
# - field string type for weakAnd
# Show that a natural way to write a query is not the fastest by presenting the query traces and compare how many documents are evaluated

In [10]:
app = vap.demo_application_package()

In [168]:
from vespa.package import Field
from vespa.package import QueryTypeField, QueryProfileType

app.get_schema("doc").add_fields(
    Field(
        name="embedding",
        type="tensor<float>(x[1])",
        indexing="attribute"
    ),
    Field(
        name="lexical",
        type="string",
        indexing="index",
        index="enable-bm25"
    ),
)

app.query_profile_type = QueryProfileType(
    fields=[
        QueryTypeField(
            name="ranking.features.query(query_embedding)",
            type="tensor<float>(x[1])"
        )
    ]
)

app.get_schema("doc").rank_profiles.pop("fields")

RankProfile('fields', '0', 'unranked', None, [Function('id', 'attribute(id)', None)], ['id'], ['id'], None, None, None, None, None, None, None, None, None, None, None)

In [169]:
#| label: schema
print(app.get_schema("doc").schema_to_text)

schema doc {
    document doc {
        field id type int {
            indexing: attribute
            attribute {
                fast-search
            }
        }
        field embedding type tensor<float>(x[1]) {
            indexing: attribute
        }
        field lexical type string {
            indexing: index
            index: enable-bm25
        }
    }
}


In [52]:
from vespa.deployment import VespaDocker

# In case running colima on macos run the following
# !sudo ln -sf $HOME/.colima/default/docker.sock /var/run/docker.sock
vespa_docker = VespaDocker(
    container_image="vespaengine/vespa:8.672.3",
)
# Start a docker container and deploy the application package
client = vespa_docker.deploy(
    application_package=app,
)

Waiting for configuration server, 0/60 seconds...
Waiting for configuration server, 5/60 seconds...
Application is up!
Finished deployment.


In [68]:
vap.redeploy(vespa_docker, app)

Deploy status code: 200


Vespa(http://localhost, 8080)

In [69]:
import random


def simulate_text():
    """
    Pics a random number of words from random numbers from 0 to 30000.
    Joins them in to a string.
    :return:
    """
    dictionary_size = 30001
    num_words = random.randint(1, 20)
    return " ".join(map(lambda n: str(n), random.sample(range(dictionary_size), num_words)))


def simulate_embedding():
    return [random.uniform(0, 1)]

In [55]:
simulate_text()

'1678 27763 4868 19582 12785 2253 25472 26042 1961 3405 1003 221 22113'

In [82]:
vap.feed(
    client=client,
    docs=[
        {
            "id": i,
            "embedding": simulate_embedding(),
            "lexical": simulate_text()
        } for i in range(100000)],
)

In [206]:
#| label: yql_base
yql_base = """
SELECT *
FROM sources *
WHERE
 (id> 1)
 AND (
   ({targetHits: 1000, approximate: false}nearestNeighbor(embedding, query_embedding))
   OR
   ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))
  )
"""
print(yql_base)


SELECT *
FROM sources *
WHERE
 (id> 1)
 AND (
   ({targetHits: 1000, approximate: false}nearestNeighbor(embedding, query_embedding))
   OR
   ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))
  )



In [186]:
request = {
    "yql": yql_base,
    "query_str": "27110 6334 10140 22335 22040 2716",
    "input.query(query_embedding)": [0.5],
    "presentation.timing": True,
    "hits": 1,
}
print(json.dumps(client.query(body=request).json, indent=2))

{
  "root": {
    "children": [
      {
        "fields": {
          "documentid": "id:doc:doc::96960",
          "sddocname": "doc"
        },
        "id": "id:doc:doc::96960",
        "relevance": 0.24059506636028924,
        "source": "test_content"
      }
    ],
    "coverage": {
      "coverage": 100,
      "documents": 100000,
      "full": true,
      "nodes": 1,
      "results": 1,
      "resultsFull": 1
    },
    "fields": {
      "totalCount": 5692
    },
    "id": "toplevel",
    "relevance": 1.0
  },
  "timing": {
    "querytime": 0.012,
    "searchtime": 0.013000000000000001,
    "summaryfetchtime": 0.0
  }
}


In [99]:
# Good we've found ~5692 docs

In [100]:
# Now let's try with tracing and ask vespa CLI to summarize the trace

In [173]:
import mycode.trace as trace

In [174]:
resp_base = client.query(body=trace.add_trace(request)).json

In [175]:
print(trace.inspect_trace(resp_base))

┌─────────┬───────────┐
│ total   │ 96.000 ms │
├─────────┼───────────┤
│ query   │ 94.000 ms │
│ summary │  1.000 ms │
│ other   │  1.000 ms │
└─────────┴───────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │     89.984 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 89.984 ms
┌───────────────┬───────────┐
│ task          │ doc[0]    │
├───────────────┼───────────┤
│ global filter │  0.000 ms │
│ ann setup     │  0.000 ms │
│ matching      │ 85.378 ms │
│ first phase   │  2.538 ms │
│ second phase  │  0.000 ms │
└───────────────┴───────────┘
looking into node doc[0]
┌───────────┬─────────────────────────────────────────────────────────────┐
│ timestamp │ event                                                       │
├───────────┼───────────────────────────────────────────

In [163]:
#| label: base_matching_summary
print(trace.get_matching_summary(trace.inspect_trace(resp_base)))

match profiling for thread #0 (total time was 91.847 ms)
┌────────┬──────────┬─────────┬──────┬────────────────────────────────────────────────┐
│ seeks  │ total_ms │ self_ms │ step │ query tree                                     │
├────────┼──────────┼─────────┼──────┼────────────────────────────────────────────────┤
│   5693 │   91.847 │   9.503 │ S    │  And[1]                                        │
│ 105690 │    2.840 │   2.628 │ S    │  ├── Attribute{int32,fs}[2] id:<range>         │
│  99998 │   77.513 │  12.364 │ N    │  ├── Or[3]                                     │
│ 100198 │    6.139 │   6.139 │ N    │  │   ├── NearestNeighbor[4]                    │
│  99998 │   59.010 │  35.133 │ N    │  │   └── WeakAnd[5]                            │
│  99998 │    3.928 │   3.906 │ N    │  │       ├── SourceBlender[6]                  │
│     37 │    0.022 │   0.022 │ N    │  │       │   └── MemoryTerm[7] lexical:27110   │
│  99998 │    3.959 │   3.944 │ N    │  │       ├── SourceBlend

In [111]:
# Above we see that weakAnd evaluated 99998 docs, which means that it can't prune matches.
# Now let's rewrite the query

In [205]:
#| label: yql_alt
yql_alt = """
SELECT *
FROM sources *
WHERE
  (id> 1 AND ({targetHits: 1000, approximate: false}
              nearestNeighbor(embedding, query_embedding))
  OR
  (id> 1 AND ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))))
"""
print(yql_alt)


SELECT *
FROM sources *
WHERE
  (id> 1 AND ({targetHits: 1000, approximate: false}
              nearestNeighbor(embedding, query_embedding))
  OR
  (id> 1 AND ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))))



In [187]:
client.query(body={
    **request,
    "yql": yql_alt,
}).json

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::96960',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::96960',
    'relevance': 0.24030457979529288,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 100000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 5692},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.007,
  'searchtime': 0.009000000000000001,
  'summaryfetchtime': 0.0}}

In [115]:
resp_alt = client.query(body={
    **trace.add_trace(request),
    "yql": yql,
}).json
print(trace.inspect_trace(resp_alt))

┌─────────┬───────────┐
│ total   │ 35.000 ms │
├─────────┼───────────┤
│ query   │ 33.000 ms │
│ summary │  1.000 ms │
│ other   │  1.000 ms │
└─────────┴───────────┘
found 1 search
┌────────┬───────┬───────────────┬───────────────┐
│ search │ nodes │ back-end time │ document type │
├────────┼───────┼───────────────┼───────────────┤
│      0 │     1 │     26.625 ms │ doc           │
└────────┴───────┴───────────────┴───────────────┘
looking into search #0
slowest node was: doc[0]: 26.625 ms
┌───────────────┬───────────┐
│ task          │ doc[0]    │
├───────────────┼───────────┤
│ global filter │  0.000 ms │
│ ann setup     │  0.000 ms │
│ matching      │ 21.827 ms │
│ first phase   │  2.696 ms │
│ second phase  │  0.000 ms │
└───────────────┴───────────┘
looking into node doc[0]
┌───────────┬─────────────────────────────────────────────────────────────┐
│ timestamp │ event                                                       │
├───────────┼───────────────────────────────────────────

In [166]:
#| label: alt_matching_summary
print(trace.get_matching_summary(trace.inspect_trace(resp_alt)))

match profiling for thread #0 (total time was 21.827 ms)
┌───────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────┐
│ seeks │ total_ms │ self_ms │ step │ query tree                                         │
├───────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────┤
│  5693 │   21.827 │   1.129 │ S    │  And[1]                                            │
│  5693 │   20.494 │   0.914 │ S    │  ├── Or[2]                                         │
│  5492 │   19.203 │   9.370 │ S    │  │   ├── And[3]                                    │
│ 99998 │    3.764 │   3.764 │ S    │  │   │   ├── Attribute{int32,fs}[4] id:<range>     │
│ 99998 │    6.070 │   6.070 │ N    │  │   │   └── NearestNeighbor[5]                    │
│   213 │    0.377 │   0.059 │ S    │  │   └── And[6]                                    │
│   213 │    0.249 │   0.073 │ S    │  │       ├── WeakAnd[7]                            │
│    38 │    0.027 │   0.013 │ S 

In [116]:
# above we see that weakAnd evaluated only 213 docs
# Which resulted in significantly lower latency: from ~100ms down to ~35ms.

In [162]:
print(trace.get_matching_summary(trace.inspect_trace(resp_alt)))

match profiling for thread #0 (total time was 21.827 ms)
┌───────┬──────────┬─────────┬──────┬────────────────────────────────────────────────────┐
│ seeks │ total_ms │ self_ms │ step │ query tree                                         │
├───────┼──────────┼─────────┼──────┼────────────────────────────────────────────────────┤
│  5693 │   21.827 │   1.129 │ S    │  And[1]                                            │
│  5693 │   20.494 │   0.914 │ S    │  ├── Or[2]                                         │
│  5492 │   19.203 │   9.370 │ S    │  │   ├── And[3]                                    │
│ 99998 │    3.764 │   3.764 │ S    │  │   │   ├── Attribute{int32,fs}[4] id:<range>     │
│ 99998 │    6.070 │   6.070 │ N    │  │   │   └── NearestNeighbor[5]                    │
│   213 │    0.377 │   0.059 │ S    │  │   └── And[6]                                    │
│   213 │    0.249 │   0.073 │ S    │  │       ├── WeakAnd[7]                            │
│    38 │    0.027 │   0.013 │ S 

In [199]:
#| label: and_query
yql_and = """
           SELECT *
           FROM sources *
           WHERE
               (id> 1)
             AND (
               ({targetHits: 1000, approximate: false}nearestNeighbor(embedding, query_embedding))
              OR
               ({targetHits: 1000, defaultIndex: "lexical", grammar: "all"}userInput(@query_str))
               )
           """
request = {
    "yql": yql_and,
    "query_str": "27110 6334 10140 22335 22040 2716",
    "input.query(query_embedding)": [0.5],
    "presentation.timing": True,
    "hits": 1,
}
client.query(body=request).json

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::96960',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::96960',
    'relevance': 0.24059506636028924,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 100000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 5493},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.007, 'searchtime': 0.008, 'summaryfetchtime': 0.0}}

In [207]:
#| label: and_query
yql_no_filters = """
          select *
          from sources *
          where (
              ({targetHits: 1000, approximate: false}nearestNeighbor(embedding, query_embedding))
             OR
              ({targetHits: 1000, defaultIndex: "lexical"}userInput(@query_str))
              )
          """
request = {
    **trace.add_trace(request),
    "yql": yql_no_filters,
    "query_str": "27110 6334 10140 22335 22040 2716",
    "input.query(query_embedding)": [0.5],
    "presentation.timing": True,
    "hits": 1,
}
resp_no_filters = client.query(body=request).json
resp_no_filters

{'root': {'children': [{'fields': {'documentid': 'id:doc:doc::96960',
     'sddocname': 'doc'},
    'id': 'id:doc:doc::96960',
    'relevance': 0.23972360666530013,
    'source': 'test_content'}],
  'coverage': {'coverage': 100,
   'documents': 100000,
   'full': True,
   'nodes': 1,
   'results': 1,
   'resultsFull': 1},
  'fields': {'totalCount': 5690},
  'id': 'toplevel',
  'relevance': 1.0},
 'timing': {'querytime': 0.018000000000000002,
  'searchtime': 0.02,
  'summaryfetchtime': 0.001},
 'trace': {'children': [{'message': "Using query profile 'default' of type 'root'"},
   {'message': 'Resolved properties:\n'},
   {'message': "Invoking chain 'vespa' [com.yahoo.prelude.statistics.StatisticsSearcher@native -> com.yahoo.prelude.querytransform.PhrasingSearcher@vespa -> ... -> federation@native]"},
   {'children': [{'message': "Invoke searcher 'com.yahoo.prelude.statistics.StatisticsSearcher in native'",
      'timestamp': 0},
     {'message': "Invoke searcher 'com.yahoo.prelude.query

In [209]:
#| label: no_filters_matching_summary
print(trace.get_matching_summary(trace.inspect_trace(resp_no_filters)))

match profiling for thread #0 (total time was 4.903 ms)
┌───────┬──────────┬─────────┬──────┬────────────────────────────────────────────────┐
│ seeks │ total_ms │ self_ms │ step │ query tree                                     │
├───────┼──────────┼─────────┼──────┼────────────────────────────────────────────────┤
│  5691 │    4.903 │   1.046 │ S    │  And[1]                                        │
│  5691 │    3.670 │   0.942 │ S    │  ├── Or[2]                                     │
│  5491 │    1.762 │   1.762 │ S    │  │   ├── NearestNeighbor[3]                    │
│   213 │    0.965 │   0.059 │ S    │  │   └── WeakAnd[4]                            │
│    38 │    0.126 │   0.009 │ S    │  │       ├── SourceBlender[5]                  │
│    37 │    0.116 │   0.116 │ S    │  │       │   └── MemoryTerm[6] lexical:27110   │
│    25 │    0.121 │   0.011 │ S    │  │       ├── SourceBlender[7]                  │
│    24 │    0.109 │   0.109 │ S    │  │       │   └── MemoryTerm[8] lexic